# Capstone Part 3: Deep Learning, MLflow and Final Checks (Google Colab)

**What this notebook does, in order:**
1. Installs the extra packages the project needs (SHAP for explanations, MLflow for experiment records).
2. Loads your project zip and checks that the tests pass.
3. Runs the whole Part 3 pipeline, including the LSTM (deep learning) model.
4. Shows the deep-learning results and the MLflow experiment records.

**How to run:** click *Runtime → Run all*. Upload `part3_github.zip` when asked (step 2).
If a cell shows an error, copy the last few lines of the error and send them back. Nothing here changes your data.

## Step 1. Install the extra packages

PyTorch is already installed in Colab. This installs SHAP (explains the deep-learning model) and MLflow (keeps experiment records).
It can take a minute or two.

In [ ]:
!pip install -q shap mlflow
import sys, torch
print("Python", sys.version.split()[0], "| PyTorch", torch.__version__)

Python 3.13.18 | PyTorch 2.11.0+cpu


## Step 2. Upload the project and unzip it

Choose `part3_github.zip` from your computer when the upload box appears.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
!unzip -q -o "{zip_name}"
%cd part3_github
print("Project files ready:", sorted(os.listdir(".")))

Saving part3_github.zip to part3_github.zip
/content/part3_github
Project files ready: ['.gitignore', 'README.md', 'data', 'outputs', 'run_all.py', 'src', 'tests']


## Step 3. Run the tests

The tests check the rules the project depends on: one row per hour, the holiday count, the outage window being excluded,
and the chronological split. All should pass (16 tests).

In [ ]:
!python -m unittest discover -s tests 2>&1 | tail -5

................
----------------------------------------------------------------------
Ran 16 tests in 0.200s

OK


## Step 4. Run the full pipeline

This trains every model, including the LSTM, and writes the results to `outputs/`.
It takes several minutes, mostly for the LSTM. The last lines should say *Pipeline finished*.

In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

In [ ]:
!python run_all.py > run_log.txt 2>&1
!tail -n 15 run_log.txt

2026-10-30 13:20:23,797 | INFO    | src.mlops | Drift WARN for 2018-08: worst PSI 0.140, MAE change -26.4%
2026-10-30 13:20:23,800 | INFO    | src.mlops | Drift WARN for 2018-09: worst PSI 0.102, MAE change -19.5%
2026-10-30 13:20:23,992 | INFO    | src.tracking | Tracked run 'drift-simulation-2018' in experiment 'traffic-mlops' [mlflow]
2026-10-30 13:20:24,205 | INFO    | src.deep_learning | LSTM windows: 22624 train, 6247 test (contiguous 24-hour sequences only)
/content/part3_github/src/deep_learning.py:80: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total += float(loss) * len(xb)
2026-10-30 13:20:31,235 | INFO    | src.deep_learning | LSTM epoch 1 training MSE 0.4190
2026-10-30 13:20:32,256 | INFO    | src.deep_learning | LSTM epoch 2 training MSE 0.1015
2026-10-30 13:20:33,323 | INFO    |

## Step 5. Check the deep-learning model

Shows the LSTM's test results (its traffic-volume error and its high-risk performance, when the project records them),
and which inputs mattered most according to the SHAP-style explanation.

In [ ]:
import pandas as pd

for name in ["lstm_metrics.csv", "lstm_surrogate_importance.csv"]:
    path = f"outputs/metrics/{name}"
    print("\n==", name)
    if os.path.exists(path):
        print(pd.read_csv(path).to_string(index=False))
    else:
        print("Not produced. Check run_log.txt for a message about deep learning.")


== lstm_metrics.csv
model   mae_2018  fidelity_r2_surrogate  persistence_mae_2018                    explain_method
 lstm 229.155451               0.994532            588.419721 SHAP (TreeExplainer on surrogate)

== lstm_surrogate_importance.csv
         Unnamed: 0  mean_abs_importance
      last_hour_cos             0.428356
last_traffic_volume             0.422781
      last_hour_sin             0.133432
    last_is_weekend             0.053375
   mean_traffic_24h             0.018003
    std_traffic_24h             0.008611
          last_temp             0.008408
    max_traffic_24h             0.005088
    last_clouds_all             0.004537
    min_traffic_24h             0.001976
       last_rain_1h             0.000415


In [ ]:
!grep -i "deep learning\|torch\|shap\|lstm\|pipeline stopped\|unexpected error" run_log.txt

2026-10-30 13:20:24,205 | INFO    | src.deep_learning | LSTM windows: 22624 train, 6247 test (contiguous 24-hour sequences only)
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
2026-10-30 13:20:31,235 | INFO    | src.deep_learning | LSTM epoch 1 training MSE 0.4190
2026-10-30 13:20:32,256 | INFO    | src.deep_learning | LSTM epoch 2 training MSE 0.1015
2026-10-30 13:20:33,323 | INFO    | src.deep_learning | LSTM epoch 3 training MSE 0.0639
2026-10-30 13:20:34,658 | INFO    | src.deep_learning | LSTM epoch 4 training MSE 0.0463
2026-10-30 13:20:35,722 | INFO    | src.deep_learning | LSTM epoch 5 training MSE 0.0371
2026-10-30 13:20:35,825 | INFO    | src.deep_learning | LSTM 2018 MAE 229.2 vs persistence 588.4
2026-10-30 13:20:47,076 | INFO    | src.deep_learning | Surrogate fidelity on 2018 (R2 against the LSTM): 0.995


In [ ]:
!grep -n "ERROR" run_log.txt
!tail -n 60 run_log.txt

2026-10-30 13:19:31,032 | INFO    | src.supervised | Regressor ridge test metrics: {'MAE': 829.635, 'RMSE': np.float64(1052.936), 'R2': 0.716}
2026-10-30 13:19:31,458 | INFO    | src.tracking | Tracked run 'regressor-ridge' in experiment 'traffic-supervised' [mlflow]
2026-10-30 13:19:40,950 | INFO    | src.supervised | Regressor random_forest test metrics: {'MAE': 224.19, 'RMSE': np.float64(384.301), 'R2': 0.962}
2026-10-30 13:19:40,977 | INFO    | src.tracking | Tracked run 'regressor-random_forest' in experiment 'traffic-supervised' [mlflow]
2026-10-30 13:19:55,522 | INFO    | src.supervised | Saved figure /content/part3_github/outputs/figures/regression_permutation_importance.png
2026-10-30 13:19:55,802 | INFO    | src.supervised | Saved figure /content/part3_github/outputs/figures/rf_week_actual_vs_predicted.png
2026-10-30 13:19:55,818 | INFO    | src.features | Chronological split: train 33998 rows (<=2017), test 6533 rows (2018); 44 rows in the 22-24 July 2016 outage window exclu

## Step 6. Check the MLflow records

MLflow keeps a list of every model run, with its settings and scores. The table below lists the runs.

In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

import mlflow
from src import config

mlflow.set_tracking_uri(config.TRACKING_DIR.as_uri())
runs = mlflow.search_runs(search_all_experiments=True)
if len(runs):
    cols = [c for c in ["experiment_id", "status", "tags.mlflow.runName"] if c in runs.columns]
    print(runs[cols].to_string())
else:
    print("No MLflow runs found. Check run_log.txt for a warning about MLflow.")

        experiment_id    status             tags.mlflow.runName
0  144014634222614588  FINISHED           drift-simulation-2018
1  103658279085184763  FINISHED               association-rules
2  103658279085184763  FINISHED                       kmeans-k7
3  275839016235616581  FINISHED        classifier-random_forest
4  275839016235616581  FINISHED  classifier-logistic_regression
5  275839016235616581  FINISHED         regressor-random_forest
6  275839016235616581  FINISHED                 regressor-ridge
